In [ ]:
!service mongodb start

In [ ]:
%env FW_CONFIG_FILE=/root/capsule/code/config/FW_config.yaml

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Electrostatic partial charges (on a molecule)

This tutorial focuses on the electrostatic partial workflow in MISPR. This is also a DFT-based workflow but is considered to be simpler in design compared to the previous binding energy workflow because it involves calculations corresponding to a single molecule instead of a set of molecules. 

As the name implies, a partial charge is the net atomic charge. Partial charges are created due to the asymmetric distribution of electrons in chemical bonds.

It is important to calculate partial charges on atoms because they give us an idea of how molecules are expected to bind, e.g. negative dipoles on a molecule will bind strongly with positive dipoles on another. Partial charges are also used in molecular dynamics simulations. 

The ESP workflow in MISPR runs the following steps:
1. Optimizes the geometry of the input structure 
2. Runs a frequency analysis on the optimized structure to make sure the molecule is a real minimum
3. Calculates the partial charges on the atoms 

Steps 1 and 2 are necessary because an optimum geometry will lead to more accurate atomic charges. 

## ESP workflow for a water molecule

Instead of running the workflow by providing the structure of the molecule, we will provide a water output file from a previous optimization calculation and will ask the workflow to skip the optimization and frequency jobs described above. 

Remember that we already optimized the structure of water in the previous binding energy workflow, so to avoid running duplicate jobs and consuming computational resources, we use the results from the previous calculation on the same molecule as input to the workflow. 

In [ ]:
from mispr.gaussian.utilities.fw_utilities import run_fake_gaussian
from mispr.gaussian.workflows.base.esp import get_esp_charges

ref_dir = "/root/capsule/data/esp/water"

wf, _ = get_esp_charges(
    mol_operation_type="get_from_gout_file",
    mol="/root/capsule/data/esp/water/water.out",
    process_mol_func=False,
    mol_name="water",
    skips=["opt", "freq"],
    esp_gaussian_inputs={
        "route_parameters": {"Density": None}},
    save_to_db=True,
    save_to_file=True,
    working_dir="/root/capsule/results/esp",
)

wf = run_fake_gaussian(
    wf,
    ref_dirs=["/root/capsule/data/esp/water/ESP"],
    input_files=["water_esp.com"],
    tolerance=10,
)

Similar to before, we will initialize the database and add the workflow to it. We'll also get a summary of the workflow. 

Notice how here we don't see any information about the optimization and frequency fireworks since we skipped those steps.

In [ ]:
from datetime import date
from fireworks import LaunchPad

today = date.today()
d = today.strftime("%Y-%m-%d")

lpad = LaunchPad.auto_load()
lpad.reset(d)
lpad.add_wf(wf)
lpad.get_wf_summary_dict(fw_id=1)

We run the workflow using rapidfire of FireWorks. 

In [ ]:
from fireworks.core.rocket_launcher import rapidfire

rapidfire(lpad)

## Output and visualization

Let us check the calculated charges on the water atoms by reading the summary json file generated by the workflow. 

In [ ]:
import json

with open("/root/capsule/results/esp/water/analysis/esp.json") as json_file:
    data = json.load(json_file)
print(data["esp"])

We can use the calculation output files to visualize the charge distribution on the molecule in the form of a heat map. We just need to feed it to a molecule visualization software like Avogadro or VMD. 

The charge distribution on the water molecules looks like this:

In [ ]:
from IPython.display import Image
Image("/root/capsule/data/esp/water/esp_water.png", width=300, height=300)

Negatively charged oxygen atom is in red while the positive hydrogens are in blue. 

# Running the ESP workflow for multiple molecules 

Here we will perform steps similar to the above, but will perform the ESP calculations for multiple molecules using a single code block instead of having to repeat the code multiple times. 

We do that by simply using a for loop and iterating over the molecules we want to calculate the ESP charges for. We will do it for three molecules, but the same thing can be done to a much bigger library.

This example illustrates how we can use MISPR to run calculations in high-throughput mode. 

## Getting the workflows

In [ ]:
esp_gaussian_inputs = {
    "functional": "B3LYP",
    "basis_set": "6-31+G*",
    "route_parameters": {
        "pop": "MK",
        "iop(6/50=1)": None,
        "NoSymmetry": None,
        "Density": None,
        "test": None,
    },
    "link0_parameters": {"%chk": "esp.chk", "%mem": "45GB", "%NProcShared": "28"},
}

for mol in ["acn", "cyclohexane", "dmso"]:
    wf, _ = get_esp_charges(
        mol_operation_type="get_from_gout_file",
        mol=f"/root/capsule/data/esp/{mol}/{mol}.out",
        process_mol_func=False,
        mol_name=mol,
        skips=["opt", "freq"],
        esp_gaussian_inputs=esp_gaussian_inputs,
        save_to_db=True,
        save_to_file=True,
        working_dir="/root/capsule/results/esp",
    )
    wf = run_fake_gaussian(
        wf,
        ref_dirs=[f"/root/capsule/data/esp/{mol}/ESP"],
        input_files=[f"{mol}_esp.com"],
        tolerance=10,
    )
    lpad.add_wf(wf)
rapidfire(lpad)

## Output and visualization

In [ ]:
for mol in ["acn", "cyclohexane", "dmso"]:
    with open(f"/root/capsule/results/esp/{mol}/analysis/esp.json") as json_file:
        data = json.load(json_file)
    print("\n", mol, "\n", data["esp"])

In [ ]:
from IPython.display import Image
Image("/root/capsule/data/esp/esp.png", width=600, height=600)